# Results

Reads the completed `compute_dp.py` output (SRVT-only + SRVT+DP) and
`compute_signatures.py` output (signatures) from `data/results/*.csv`,
builds distance matrices, plots MDS for each method, and scores them with
1-NN accuracy and silhouette. No heavy computation happens here -- only
reading and plotting. Run `compute_dp.py` and `compute_signatures.py`
first if the CSVs you want don't exist yet.

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parent  # one level up from notebooks/
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from compute_dp import ANIMATIONS

## Load a results CSV

Change `RESULTS_FILE` to compare a different run (e.g. a different depth)
-- each run should be its own file, per the resumable-CSV-per-run design.

In [ ]:
DP_RESULTS_FILE = PROJECT_ROOT / "data/results/dp_depth6.csv"
SIG_RESULTS_FILE = PROJECT_ROOT / "data/results/signatures.csv"

df_dp = pd.read_csv(DP_RESULTS_FILE)
df_sig = pd.read_csv(SIG_RESULTS_FILE)
print(f"{len(df_dp)} DP rows, {len(df_sig)} signature rows")
df_dp.head()

## Build the distance matrices

In [ ]:
ids = sorted(ANIMATIONS)
idx = {aid: i for i, aid in enumerate(ids)}
n = len(ids)
labels = np.array([ANIMATIONS[aid]["label"] for aid in ids])

D_srvt = np.zeros((n, n))
D_dp = np.zeros((n, n))
D_sig = np.zeros((n, n))

for _, row in df_dp.iterrows():
    i, j = idx[int(row.id1)], idx[int(row.id2)]
    D_srvt[i, j] = D_srvt[j, i] = row.distance
    D_dp[i, j] = D_dp[j, i] = row.dp_distance

for _, row in df_sig.iterrows():
    i, j = idx[int(row.id1)], idx[int(row.id2)]
    D_sig[i, j] = D_sig[j, i] = row.signature_distance

print("matrices built:", D_srvt.shape, D_dp.shape, D_sig.shape)

## Classical MDS and plotting

In [ ]:
def classical_mds(D, k=2):
    D2 = D ** 2
    n_ = D2.shape[0]
    J = np.eye(n_) - np.ones((n_, n_)) / n_
    B = -0.5 * J @ D2 @ J
    w, v = np.linalg.eigh(B)
    order = np.argsort(w)[::-1]
    w, v = w[order], v[:, order]
    L = np.diag(np.sqrt(np.clip(w[:k], 0, None)))
    return v[:, :k] @ L

COLORS = {"walk": "teal", "run/jog": "red", "forward jump": "orange"}

def plot_mds(D, title, seed=0):
    coords = classical_mds(D)
    rng = np.random.default_rng(seed)
    coords_j = coords + rng.normal(0, 0.012 * (coords.std() + 1e-9), coords.shape)

    fig, ax = plt.subplots(figsize=(7, 6))
    for label in COLORS:
        mask = labels == label
        ax.scatter(coords_j[mask, 0], coords_j[mask, 1], color=COLORS[label], label=label, s=60)
    ax.set_title(title)
    ax.legend()
    ax.set_xticks([]); ax.set_yticks([])
    plt.show()
    return coords

In [ ]:
_ = plot_mds(D_srvt, "SRVT only (no DP)")
_ = plot_mds(D_dp, "SRVT + DP")
_ = plot_mds(D_sig, "Signature")

## 1-NN accuracy and silhouette score

In [ ]:
def score(D):
    nn_correct = []
    sil = []
    for i in range(n):
        order = np.argsort(D[i])
        nn = order[1]  # nearest neighbor, excluding self at order[0]
        nn_correct.append(labels[nn] == labels[i])

        same = (labels == labels[i]).copy()
        same[i] = False
        a = D[i][same].mean()
        others = set(labels) - {labels[i]}
        b = min(D[i][labels == lbl].mean() for lbl in others)
        sil.append((b - a) / max(a, b))
    return np.mean(nn_correct), np.mean(sil)

for name, D in [("SRVT only", D_srvt), ("SRVT + DP", D_dp), ("Signature", D_sig)]:
    nn_acc, sil = score(D)
    print(f"{name:12s}  1-NN={nn_acc:.2f}  silhouette={sil:.2f}")

## Find misclassified points

For each animation, check whether its single nearest neighbor (by
distance, not by the 2D MDS plot -- MDS can visually mislead) shares its
label.

In [ ]:
def find_misclassified(D, name):
    print(f"--- {name} ---")
    for i in range(n):
        order = np.argsort(D[i])
        nn = order[1]
        if labels[nn] != labels[i]:
            aid, nn_aid = ids[i], ids[nn]
            print(f"  {ANIMATIONS[aid]['file']} ({labels[i]}) "
                  f"-> nearest: {ANIMATIONS[nn_aid]['file']} ({labels[nn]}), dist={D[i,nn]:.3f}")

find_misclassified(D_srvt, "SRVT only")
find_misclassified(D_dp, "SRVT + DP")
find_misclassified(D_sig, "Signature")

## Comparing two runs (e.g. different depths)

Load a second CSV the same way and compare side by side, now that each
run is saved to its own file instead of overwriting a shared database.

In [ ]:
# Example: uncomment once you have more than one DP results file
# df_dp9 = pd.read_csv("data/results/dp_depth9.csv")
# D_dp9 = np.zeros((n, n))
# for _, row in df_dp9.iterrows():
#     i, j = idx[int(row.id1)], idx[int(row.id2)]
#     D_dp9[i, j] = D_dp9[j, i] = row.dp_distance
# print("depth 6:", score(D_dp))
# print("depth 9:", score(D_dp9))